# NIfTI to Atix
Este notebook permite convertir volumenes y segmentaciones al formato DICOM extendido usado por Atix para visualizarlas directamente en Atix

# Importación de librerías y funciones auxiliares

In [ ]:
import SimpleITK as sitk
import os
import matplotlib.pyplot as plt
import pydicom
import numpy as np
from datetime import datetime

In [ ]:
images_path = "C:/Users/jmriv/Documents/GitHub Repos/AISD/dicom/0019983/CT"

In [ ]:
dcm_files = []
for f in os.listdir(images_path):
    if '.dcm' in f:
        dcm_files.append(pydicom.dcmread(os.path.join(images_path, f)))

In [ ]:
reader = sitk.ImageSeriesReader()
series_ids = reader.GetGDCMSeriesIDs(images_path)
series_files = reader.GetGDCMSeriesFileNames(images_path, series_ids[0])
reader.SetFileNames(series_files)

In [ ]:
img = reader.Execute()

In [ ]:
sitk.WriteImage(img, f"{images_path}/pruebaNiFti.nii.gz")

In [ ]:
print(img.GetSize())
print(img.GetSpacing())
print(img.GetOrigin())
print(img.GetDirection())

In [ ]:
segmentation_folder = "C:/Users/jmriv/Documents/demoTotalSegmentator"

In [ ]:
label = sitk.ReadImage(f"{segmentation_folder}/caudate_nucleus.nii.gz", sitk.sitkUInt8)

In [ ]:
print(label.GetSize())
print(label.GetSpacing())
print(label.GetOrigin())
print(label.GetDirection())

In [ ]:
z = img.GetSize()[2] // 2

img_np = sitk.GetArrayFromImage(img)
lbl_np = sitk.GetArrayFromImage(label)

plt.imshow(img_np[z], cmap="gray")
plt.imshow(lbl_np[z], alpha=0.4, cmap="jet")
plt.title("DICOM + TotalSegmentator label")
plt.axis("off")
plt.show()


In [ ]:
label_np = sitk.GetArrayFromImage(label)

In [ ]:
mask_dict = {}

for z, dicom_path in enumerate(series_files):
    # Read metadata for this single DICOM file
    ds = sitk.ReadImage(dicom_path)
    sop_uid = ds.GetMetaData("0008|0018")  # SOPInstanceUID

    # Get corresponding mask slice
    mask_slice = label_np[z]              # (Y, X)
    if not (mask_slice == 0).all():
        mask_1d = mask_slice.flatten().astype(np.uint8)

        mask_dict[sop_uid] = mask_1d


In [ ]:
mask_dict

# Revisión de multi-label por pixel

In [ ]:
from pathlib import Path

label_dir = Path(segmentation_folder)
label_files = sorted(label_dir.glob("*.nii.gz"))

assert len(label_files) == 16, "Expected 16 label files"


In [ ]:
label_arrays = []
label_names = []

for lf in label_files:
    lbl = sitk.ReadImage(str(lf), sitk.sitkUInt8)

    arr = sitk.GetArrayFromImage(lbl)  # (Z, Y, X)
    label_arrays.append(arr)
    label_names.append(lf.stem)


In [ ]:
label_stack = np.stack(label_arrays, axis=0)  # (N_labels, Z, Y, X)

# Sum across labels
overlap_map = np.sum(label_stack, axis=0)     # (Z, Y, X)

# Voxels where more than one label is present
overlap_voxels = overlap_map > 1
num_overlap_voxels = np.count_nonzero(overlap_voxels)


In [ ]:
print(f"Overlapping voxels: {num_overlap_voxels}")

# Arreglo consolidado

In [ ]:
label_arrays = []
label_names = []

for lf in label_files:
    lbl = sitk.ReadImage(str(lf), sitk.sitkUInt8)

    arr = sitk.GetArrayFromImage(lbl)  # (Z, Y, X)
    label_arrays.append(arr)
    label_names.append(lf.stem)

In [ ]:
label_names

In [ ]:
label_arrays[0] = np.zeros(label_arrays[0].shape)
label_arrays[2] = np.zeros(label_arrays[0].shape)
label_arrays[3] = np.zeros(label_arrays[0].shape)
label_arrays[4] = np.zeros(label_arrays[0].shape)
label_arrays[8] = np.zeros(label_arrays[0].shape)
label_arrays[9] = np.zeros(label_arrays[0].shape)
label_arrays[10] = np.zeros(label_arrays[0].shape)
label_arrays[11] = np.zeros(label_arrays[0].shape)
label_arrays[12] = np.zeros(label_arrays[0].shape)
label_arrays[13] = np.zeros(label_arrays[0].shape)
label_arrays[14] = np.zeros(label_arrays[0].shape)
label_arrays[15] = np.zeros(label_arrays[0].shape)

In [ ]:
(label_arrays[0] == 0).all()

In [ ]:
temp_arrays = []
for i, array in enumerate(label_arrays):
    array[array==1] = i+1
    if len(temp_arrays) == 0:
        temp_arrays = array
    else:
        temp_arrays += array

In [ ]:
z = img.GetSize()[2]-3

img_np = sitk.GetArrayFromImage(img)
lbl_np = sitk.GetArrayFromImage(label)

plt.imshow(img_np[z], cmap="gray")
plt.imshow(temp_arrays[z], alpha=0.4, cmap="jet")
plt.title("DICOM + TotalSegmentator label")
plt.axis("off")
plt.show()


In [ ]:
mask_dict = {}
for z, dicom_path in enumerate(series_files):
    # Read metadata for this single DICOM file
    ds = sitk.ReadImage(dicom_path)
    sop_uid = ds.GetMetaData("0008|0018")  # SOPInstanceUID
    mask_slice = temp_arrays[z]
    print((mask_slice == 0).all())
    if not (mask_slice == 0).all():
        mask_1d = mask_slice.flatten().astype(np.uint16)

        mask_dict[sop_uid] = mask_1d

In [ ]:
for _, i in mask_dict.items():
    print(np.unique(i))

In [ ]:
colors = [
    [220,20,60],
    [34,139,34],
    [255,165,0],
    [30,144,255],
    [148,0,211],
    [255,215,0],
    [0,206,209],
    [255,20,147],
    [139,69,19],
    [0,0,205],
    [46,139,87],
    [178,34,34],
    [72,61,139],
    [154,205,50],
    [199,21,133],
    [60,180,75]
]


In [ ]:
layer_values = {}
for SUID, mask in mask_dict.items():
    slice_dict = {}
    for values in np.unique(mask):
        if values > 0:
            slice_dict[values] = {
                'code': values,
                'name': label_names[values-1],
                'label': label_names[values-1],
                'avgIntensity': 0,
                'rgb': colors[values-1]
            }
    layer_values[SUID] = slice_dict

# Archivo DICOM para Atix

In [ ]:
ORGANIZATION_PREFIX = "1.2.826.0.1.3680043.10.718"
APPLICATION_ID = "1"
APPLICATION_VERSION_ID = "1"

def generateFormatedDate(date):
    month = "0" + str(date.month) if date.month < 10 else str(date.month)
    day = "0" + str(date.day) if date.day < 10 else str(date.day)
    return str(date.year) + month + day        

def generateFormatedTime(date):
    hour = "0" + str(date.hour) if date.hour < 10 else str(date.hour)
    minute = "0" + str(date.minute) if date.minute < 10 else str(date.minute)
    second = "0" + str(date.second) if date.second < 10 else str(date.second)
    return hour + minute + second
    
def generate_uid():
    """
    Function to generate a uid for the DICOM that is going to be processed
    :return: the obtained uid
    """
    date = datetime.now()
    date_stringified = (
        str(date).replace("-", "").replace(":", "").replace(" ", "").replace(".", "")
    )
    uid = f"{ORGANIZATION_PREFIX}.{APPLICATION_ID}.{APPLICATION_VERSION_ID}.{date_stringified}"
    return uid

In [ ]:
date = datetime.now()
formated_date = generateFormatedDate(date)
formated_time = generateFormatedTime(date)

In [ ]:
algoritmo = "TotalSegmentator"
segmentation_name = "Demo"
segmentation_description = f"{segmentation_name} {algoritmo} {formated_date}"

In [ ]:
series_uid = generate_uid()
instanceUID_list = []
output = {}

In [ ]:
def applyWindowLevel(manualHU, WC, WW):
  windowedImage = manualHU.copy()
  windowedImage[windowedImage < (WC-(WW/2))] = (WC-(WW/2))
  windowedImage[windowedImage > (WC+(WW/2))] = (WC+(WW/2))
  return windowedImage

In [ ]:
PACS_ORIGINAL_PATH = images_path
for path in os.listdir(images_path):
    originalImg = pydicom.dcmread(
                os.path.join(PACS_ORIGINAL_PATH, path), force=True
                )


    originalImg.AcquisitionDate = formated_date
    originalImg.AcquisitionTime = formated_time
    originalImg.SeriesNumber = 2
    originalImg.SeriesDescription = segmentation_description
    originalImg.SeriesInstanceUID = series_uid
    # Se guardan los valores de la máscara en un campo nuevo
    if originalImg.SOPInstanceUID in mask_dict:
        mask = mask_dict[originalImg.SOPInstanceUID]
        print(np.unique(mask))
        print(mask.shape)
        originalImg.add_new([0x0062, 0x0014], "OW", mask.tobytes())
        # En caso de que se esté pintando una imágen de ceros se indica que es una segmentación
        originalImg.add_new([0x0062, 0x0001], "CS", "1")
        # Se añade el nombre del algoritmo
        originalImg.add_new([0x0062, 0x0009], "LO", algoritmo)
        # Los bits que se almacenan no son signed, por lo que el campo 0028,0103 debe ser 0
        originalImg.PixelRepresentation = 0

        # Para cada región de interés se creará un bloque, que se guardará en una secuencia
        segmentationSequence = []
        for key, value in layer_values[originalImg.SOPInstanceUID].items():
            bloque = pydicom.dataset.Dataset()
            bloque.add_new(0x00620004, "US", value.get('code') )
            bloque.add_new(0x00620005, "LO", value.get('name'))
            bloque.add_new(0x00620006, "ST", value.get('label'))
            bloque.add_new(0x00620022, "DS", value.get('avgIntensity'))
            bloque.add_new(0x00620013, "IS", value.get('rgb'))
            segmentationSequence.append(bloque)

        segmentationSequence = pydicom.sequence.Sequence(segmentationSequence)
        originalImg.add_new([0x0062, 0x0002], "SQ", segmentationSequence)

        img_np = applyWindowLevel(originalImg.pixel_array*originalImg.RescaleSlope+originalImg.RescaleIntercept, 32,32)
        lbl_np = mask.reshape(512, 512)

        plt.imshow(img_np, cmap="gray")
        plt.imshow(lbl_np, alpha=0.4, cmap="jet")
        plt.title("DICOM + TotalSegmentator label")
        plt.axis("off")
        plt.show()

    originalImg.SOPInstanceUID = generate_uid()
    instanceUID_list.append(originalImg.SOPInstanceUID)
    originalImg.is_implicit_VR = False
    originalImg.file_meta.TransferSyntaxUID = "1.2.840.10008.1.2.1"
    originalImg.save_as(
        os.path.join(f'{segmentation_folder}/AtixDicom', path), write_like_original=False
        )